In [47]:
%pip install shap --quiet

Note: you may need to restart the kernel to use updated packages.


In [48]:
import joblib
import shap
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

In [49]:
X_test = pd.read_csv(r"../data/clean/X_test.csv")

In [50]:
model_path = Path("../artifacts/final_model.pkl")
model = joblib.load(model_path)

In [51]:
preprocessor = model.named_steps["features"]
estimator = model.named_steps["model"]

In [52]:
X_transformed = preprocessor.transform(X_test)

C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [53]:
ct = model.named_steps["features"].named_steps["preprocessing"]

feature_names = ct.get_feature_names_out()

In [54]:
explainer = shap.Explainer(estimator, X_transformed)
shap_values = explainer(X_transformed)

In [55]:
importance = np.abs(shap_values.values).mean(axis=0)

df_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False)

print(df_importance.head(20))

                   feature  importance
4         num__OverallQual    0.051816
6           num__YearBuilt    0.047963
16          num__GrLivArea    0.038195
5         num__OverallCond    0.031769
38            num__TotalSF    0.031183
14           num__2ndFlrSF    0.026102
42        nom__MSZoning_RL    0.025159
27         num__GarageArea    0.024666
13           num__1stFlrSF    0.024496
122  nom__RoofMatl_CompShg    0.023317
116   nom__RoofStyle_Gable    0.021276
1          num__MSSubClass    0.020229
165  nom__Foundation_PConc    0.019047
118     nom__RoofStyle_Hip    0.018751
39   nom__MSZoning_C (all)    0.018624
9          num__BsmtFinSF1    0.017466
225        ord__Functional    0.016961
218        ord__GarageQual    0.016749
17       num__BsmtFullBath    0.015039
7        num__YearRemodAdd    0.014813


In [56]:
# para guardar las más importantes y usarlas en la página de streamlit
import re

def get_original_feature(name):
    name = name.split("__")[-1]  # quitar prefijo (num__, nom__, etc.)
    
    # eliminar categoría en one-hot
    if "_" in name:
        return name.split("_")[0]
    
    return name

df_importance["original_feature"] = df_importance["feature"].apply(get_original_feature)

# agrupar
grouped = df_importance.groupby("original_feature")["importance"] \
    .sum() \
    .sort_values(ascending=False)

print(grouped.head(30))

top_30_features = grouped.head(30).index.tolist()

original_feature
Neighborhood     0.068265
MSZoning         0.062709
OverallQual      0.051816
YearBuilt        0.047963
Exterior1st      0.046382
RoofStyle        0.041914
Exterior2nd      0.041108
RoofMatl         0.040914
GrLivArea        0.038195
OverallCond      0.031769
TotalSF          0.031183
2ndFlrSF         0.026102
GarageArea       0.024666
1stFlrSF         0.024496
Foundation       0.024247
SaleType         0.021253
MSSubClass       0.020229
LotConfig        0.019738
HouseStyle       0.018958
SaleCondition    0.018607
BsmtFinSF1       0.017466
Functional       0.016961
GarageQual       0.016749
BsmtFullBath     0.015039
YearRemodAdd     0.014813
Fireplaces       0.014793
HalfBath         0.014048
Condition1       0.012727
TotalBsmtSF      0.012444
TotRmsAbvGrd     0.012394
Name: importance, dtype: float64


In [57]:
df_grouped = grouped.reset_index()
df_grouped.columns = ["feature", "importance"]

In [58]:
import joblib

# top 30 (para inputs)
joblib.dump(top_30_features, "../artifacts/top_30_features.pkl")

# ranking completo (para visualización)
joblib.dump(df_grouped, "../artifacts/shap_global_importance.pkl")

['../artifacts/shap_global_importance.pkl']

In [59]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

top_plot = df_grouped.head(20)[::-1]

plt.barh(top_plot["feature"], top_plot["importance"])
plt.xlabel("SHAP importance")
plt.title("Top variables más importantes (SHAP)")

plt.tight_layout()
plt.savefig("../artifacts/shap_importance.png")
plt.close()